# Assignment 6

In [4]:
import os

import random
import keras
from keras import layers, ops
from keras.layers import TextVectorization

import numpy as np
import tensorflow as tf

import urllib.request
import zipfile

import re
import string


In [5]:
"""
url = "https://www.manythings.org/anki/fin-eng.zip"
zip_path = "fin-eng.zip"
extract_dir = "fin-eng"

if not os.path.exists(f"{extract_dir}/fin.txt"):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Referer": "https://www.manythings.org/anki/"
    }
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as response, open(zip_path, "wb") as out_file:
        out_file.write(response.read())
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

print("Extracted files:", os.listdir(extract_dir))
"""

'\nurl = "https://www.manythings.org/anki/fin-eng.zip"\nzip_path = "fin-eng.zip"\nextract_dir = "fin-eng"\n\nif not os.path.exists(f"{extract_dir}/fin.txt"):\n    headers = {\n        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",\n        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",\n        "Accept-Language": "en-US,en;q=0.5",\n        "Referer": "https://www.manythings.org/anki/"\n    }\n    req = urllib.request.Request(url, headers=headers)\n    with urllib.request.urlopen(req) as response, open(zip_path, "wb") as out_file:\n        out_file.write(response.read())\n    with zipfile.ZipFile(zip_path, "r") as zip_ref:\n        zip_ref.extractall(extract_dir)\n\nprint("Extracted files:", os.listdir(extract_dir))\n'

## Parsin the Data

In [6]:
text_file = "fin-eng/fin.txt"

with open(text_file, encoding="utf-8") as f:
    lines = f.read().split("\n")[:-1]

text_pairs = []
for line in lines:
    english, finnish, rest = line.split("\t")
    finnish = "[start] " + finnish + " [end]"
    text_pairs.append((english, finnish))

for _ in range(5):
    print(random.choice(text_pairs))

('Tom and Mary went canoeing together.', '[start] Tuomas ja Mirjami menivät melomaan yhdessä. [end]')
('I saw Tom earlier.', '[start] Näin Tomin aikaisemmin. [end]')
('Are you seriously thinking about going?', '[start] Ajatteletteko vakavissanne lähtemistä? [end]')
('How did you make this soup?', '[start] Miten olet tehnyt tämän keiton? [end]')
('I like this one better.', '[start] Mä tykkään tästä enemmän. [end]')


Split the sentence parts into a **training set, validation set and test set.**

In [7]:
random.shuffle(text_pairs)
num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples : num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples :]

print(f"{len(text_pairs)} total pairs")
print(f"{len(train_pairs)} training pairs")
print(f"{len(val_pairs)} validation pairs")
print(f"{len(test_pairs)} test pairs")

73604 total pairs
51524 training pairs
11040 validation pairs
11040 test pairs


As shown in the results, we end up with **total pairs of 73604** that are divided into three:

- Training pairs: **51524**,
- Validation pairs: **11040**, 
- Test pairs: **11040**.

## Vectorizing the Text data

Use **two instances** of the `TextVectorixation` layer to vectorize the text data to turn the **original strings** into **integer sequences** where each integer represents the index of a word in a vocabulary.

One is for the English language and one for the Finnish language.

In [8]:
strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

vocab_size = 15000
sequence_length = 20
batch_size = 64

In [9]:
def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(lowercase, '[%s]' % re.escape(strip_chars), '')

eng_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

fin_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)

train_eng_texts = [pair[0] for pair in train_pairs]
train_fin_texts = [pair[1] for pair in train_pairs]
eng_vectorization.adapt(train_eng_texts)
fin_vectorization.adapt(train_fin_texts)

## Format datasets

During each **training step**, each **model** will seek to predict target words **N+1** (and beyond) using the source sentence and the target words **0 to N**.

In [13]:
def format_dataset(eng, fin):
    eng = eng_vectorization(eng)
    fin = fin_vectorization(fin)

    return(
        {
            "encoder_inputs": eng,
            "decoder_inputs": fin[:, :-1],
        },
        fin[:, 1:],
    )

In [14]:
def make_dataset(pairs):
    eng_texts, fin_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    fin_texts = list(fin_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, fin_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset)

    return dataset.cache().shuffle(2048).prefetch(16)

In [15]:
train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [16]:
for inputs, targets in train_ds.take(1):
    print(f'inputs["encoder_inputs"].shape: {inputs["encoder_inputs"].shape}')
    print(f'inputs["decoder_inputs"].shape: {inputs["decoder_inputs"].shape}')
    print(f"targets.shape: {targets.shape}")

inputs["encoder_inputs"].shape: (64, 20)
inputs["decoder_inputs"].shape: (64, 20)
targets.shape: (64, 20)


As shown above, there are batches of **64 pairs** and all sequences are **20 steps long**.

## Building the Model

The **sequence-to-sequence Transformer** consist of a `TransformerEncoder` and a `TransformerDecoder` chained together. 

`PositionalEmbedding` layer is also used to make the **Model** aware of the word order.

The **source sequence** is passed to the `TransformerEncoder` which will produce a new representation on it. The new representation will then be passed to the `TransformerDecoder` together with the **target sequence** so far *(target words 0 to N)*. The `TransformerDecoder` will then seek to predict the next words in the **target sequence** *(N+1 and beyond)*.

In [17]:
import keras.ops as ops

In [23]:
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        if mask is not None:
            padding_mask = ops.cast(mask[:, None, :], dtype="int32")
        else:
            padding_mask = None

        attention_output = self.attention(
            query=inputs, value=inputs, key=inputs, attention_mask=padding_mask
        )
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
            }
        )
        return config


class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim
        )
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        length = ops.shape(inputs)[-1]
        positions = ops.arange(0, length, 1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return ops.not_equal(inputs, 0)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config


class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, latent_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.latent_dim = latent_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.attention_2 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(latent_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, encoder_outputs, mask=None):
        attention_output_1 = self.attention_1(
            query=inputs,
            value=inputs,
            key=inputs,
            use_causal_mask=True,
        )
        out_1 = self.layernorm_1(inputs + attention_output_1)

        if mask is not None:
            padding_mask = ops.cast(mask[:, None, :], dtype="int32")
        else:
            padding_mask = None

        attention_output_2 = self.attention_2(
            query=out_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=padding_mask,
        )
        out_2 = self.layernorm_2(out_1 + attention_output_2)

        proj_output = self.dense_proj(out_2)
        return self.layernorm_3(out_2 + proj_output)

    def get_causal_attention_mask(self, inputs):
        input_shape = ops.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = ops.arange(sequence_length)[:, None]
        j = ops.arange(sequence_length)
        mask = ops.cast(i >= j, dtype="int32")
        mask = ops.reshape(mask, (1, input_shape[1], input_shape[1]))
        mult = ops.concatenate(
            [ops.expand_dims(batch_size, -1), ops.convert_to_tensor([1, 1])],
            axis=0,
        )
        return ops.tile(mask, mult)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "latent_dim": self.latent_dim,
                "num_heads": self.num_heads,
            }
        )
        return config

Next, assemble the **end-to-end Model**. 

In [19]:
embed_dim = 256
latent_dim = 2048
num_heads = 8

In [24]:
# Encoder
encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="encoder_inputs")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = TransformerEncoder(embed_dim, latent_dim, num_heads)(x)
encoder = keras.Model(encoder_inputs, encoder_outputs)

# Decoder (uses encoded_seq_inputs)
decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="decoder_inputs")
encoded_seq_inputs = keras.Input(shape=(None, embed_dim), name="decoder_state_inputs")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
x = TransformerDecoder(embed_dim, latent_dim, num_heads)(x, encoded_seq_inputs)
x = layers.Dropout(0.5)(x)
decoder_outputs = layers.Dense(vocab_size, activation="softmax")(x)
decoder = keras.Model([decoder_inputs, encoded_seq_inputs], decoder_outputs)

# End-to-end Transformer (connects encoder_outputs into decoder)
decoder_outputs = decoder([decoder_inputs, encoder_outputs])
transformer = keras.Model(
    {"encoder_inputs": encoder_inputs, "decoder_inputs": decoder_inputs},
    decoder_outputs,
    name="transformer",
)

c:\Users\araz3\anaconda3\envs\nlp_env\Lib\site-packages\keras\src\layers\layer.py:1039: UserWarning: Layer 'functional_7' (of type Functional) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


## Training the Model

In [25]:
epochs = 1  # Should be at least 30 for convergence

transformer.summary()
transformer.compile(
    "rmsprop",
    loss=keras.losses.SparseCategoricalCrossentropy(ignore_class=0),
    metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=epochs, validation_data=val_ds)


Model: "transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 256) │  3,845,120 │ encoder_inputs[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, None, 256) │  3,155,456 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_7        │ (None, None,      │ 12,959,640 │ decoder_inputs[0… │
│ (Functional)        │ 15000)            │            │ transformer_enco… │
│                     │                   │            │ not_equal_4[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 19,960,216 (76.14 MB)

 Trainable params: 19,960,216 (76.14 MB)

 Non-trainable params: 0 (0.00 B)

c:\Users\araz3\anaconda3\envs\nlp_env\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


806/806 ━━━━━━━━━━━━━━━━━━━━ 1033s 1s/step - accuracy: 0.3027 - loss: 5.0490 - val_accuracy: 0.3957 - val_loss: 3.7887


## Decoding test sentences

Demonstration of translating brand new English sentences can begin. Simply feed into the model the **vectorized** English sentence as well as the target token `"[start]"` and then repeatedly generate the next token until it its the token `"[end]"`.

In [27]:
fin_vocab = fin_vectorization.get_vocabulary()
fin_index_lookup = dict(zip(range(len(fin_vocab)), fin_vocab))
max_decoded_sentence_length = 20

In [28]:
def decode_sequence(input_sentence):
    tokenized_input_sentence = eng_vectorization([input_sentence])
    decoded_sentence = "[start]"

    for i in range(max_decoded_sentence_length):
        tookenized_target_sentence = fin_vectorization([decoded_sentence])[:, :-1]
        predictions = transformer(
            {
                "encoder_inputs": tokenized_input_sentence,
                "decoder_inputs": tookenized_target_sentence,
            },
        )

        # ops.argmax(predictions[0, i, :]) is not a concrete value for jax here
        sampled_token_index = ops.convert_to_numpy(
            ops.argmax(predictions[0, i, :])
        ).item(0)
        sampled_token = fin_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token

        if sampled_token == "[end]":
            break
    
    return decoded_sentence

In [32]:
test_eng_texts = [pair[0] for pair in test_pairs]

for _ in range(10):
    input_sentence = random.choice(test_eng_texts)
    translated = decode_sequence(input_sentence)
    print(f"English: {input_sentence}")
    print(f"Finnish: {translated}")
    print("-" * 40)

English: Where are my things?
Finnish: [start] missä minun [UNK] [end]
----------------------------------------
English: I've left my charger at home.
Finnish: [start] minä olen [UNK] kotona kotiin [end]
----------------------------------------
English: Tom may be dangerous.
Finnish: [start] tom voi olla [UNK] [end]
----------------------------------------
English: I think we're going to regret this.
Finnish: [start] minusta [UNK] [UNK] [end]
----------------------------------------
English: If you translate from your second language into your own native language, rather than the other way around, you're less likely to make mistakes.
Finnish: [start] jos [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK]
----------------------------------------
English: I don't want a party on my birthday.
Finnish: [start] en halua [UNK] [UNK] [end]
----------------------------------------
English: How cute!
Finnish: [start] kuinka [UNK] [e